In [ ]:
#import
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../..')


import numpy as np
import src.demo as demo
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib

import itertools

In [ ]:

# Load the pointcloud files (for visualization purposes)
save_name = '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/init_scene_pcls.npz'
scene_pcls = np.load(save_name)
scene_pcls = {k: scene_pcls[k] for k in scene_pcls.keys()}

# Load recorded demo transformation
transform_name = '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/ee_transform.npz'
demo_transform = np.load(transform_name)

init_transform = utils.pos_quat_to_transform(demo_transform['init_pos'],
                                             demo_transform['init_quat'])

final_transform = utils.pos_quat_to_transform(demo_transform['final_pos'],
                                              demo_transform['final_quat'])

ee_transform = np.matmul(final_transform, np.linalg.inv(init_transform))

# Load the warp reconstructions
warps = np.load("/home/rthomp12/fewshot/blue_mug_thin_rack_demo/initial_scene_warps.npz", allow_pickle=True)
warps = {k: warps[k] for k in warps.keys()}

child_params = warps['child_params'].item()
parent_params = warps['parent_params'].item()
child_reconstruction = warps['child_reconstructions'].item()

In [9]:
#Set the object names and canon files that we're using

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Mug v Rack
parent_part_names = ['trunk', 'branch']
child_part_names = ['cup', 'handle']


parent_part_model_files = {'trunk': '/home/rthomp12/fewshot/part_based_warp_models/trunk_dict_20240412-042732', 
                           'branch': '/home/rthomp12/fewshot/part_based_warp_models/branch_dict_20240412-042732'}

child_part_model_files = {'cup': '/home/rthomp12/fewshot/part_based_warp_models/cup_dict_20240202-160637',
                          'handle': '/home/rthomp12/fewshot/part_based_warp_models/handle_dict_20240202-160637'}


# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Teapot v Mug

# child_part_names = ['body', 'lid', 'tea_handle', 'spout']
# parent_part_names = ['cup', 'handle']


# child_part_model_files = {'body': '/home/rthomp12/fewshot/part_based_warp_models/trunk_dict_20240412-042732', 
#                           'tea_handle': '/home/rthomp12/fewshot/part_based_warp_models/branch_dict_20240412-042732',
#                           'lid': ,
#                           'spout': ,}
# parent_part_model_files = {'cup': '/home/rthomp12/fewshot/part_based_warp_models/cup_dict_20240202-160637',
#                           'handle': '/home/rthomp12/fewshot/part_based_warp_models/handle_dict_20240202-160637'}




# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Mug v Bowl

# child_part_names = ['bowl']
# parent_part_names = ['cup', 'handle']


# child_part_model_files = {'bowl': }
# parent_part_model_files = {'cup': '/home/rthomp12/fewshot/part_based_warp_models/cup_dict_20240202-160637',
#                           'handle': '/home/rthomp12/fewshot/part_based_warp_models/handle_dict_20240202-160637'}


parent_part_models = {part: CanonPart.from_pickle(parent_part_model_files[part]) for part in parent_part_names}
child_part_models = {part: CanonPart.from_pickle(child_part_model_files[part]) for part in child_part_names}



In [10]:
# Verify the reconstruction and demo transformation

child_params = warps['child_params'].item()
reconstructions = {}
transformed_scene_pcls = {}

for child_part in child_part_names:
    child_transform = utils.pos_quat_to_transform(child_params[child_part].position, child_params[child_part].quat)
    new_child_transform = np.matmul(ee_transform, child_transform)
    child_params[child_part].position, child_params[child_part].quat = \
        utils.transform_to_pos_quat(new_child_transform)


    transformed_scene_pcls[child_part] = utils.transform_pcd(scene_pcls[child_part],
                                                 ee_transform,
                                               )
    reconstructions[f'reconstructed_{child_part}'] = \
        child_part_models[child_part].to_transformed_pcd(child_params[child_part])

viz_utils.show_pcds_plotly(transformed_scene_pcls|reconstructions)

    

In [11]:
# Find and save interaction points 

nearby_points_delta = 0.035 # Empirically picked

(
    knns,
    deltas,
    target_indices,
) = demo.save_place_nearby_points_by_parts_v2(
    child_part_names,
    child_part_models,
    child_params,
    parent_part_names,
    parent_part_models,
    parent_params,
    nearby_points_delta,
)
knn_pickle_file = '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/interaction_points.pkl'

interaction_points = {'knns': knns, 'deltas': deltas, "target_indices": target_indices}
pickle.dump(interaction_points, open(knn_pickle_file, 'wb'))


@@ 0.5590190218332428
@@ 0.6164104972867747
@@ 0.5839277987609472
@@ 0.654618962263681


In [ ]:
# Visualize interaction points 

targets_child = {part: {} for part in child_part_names}
targets_parent = {part: {} for part in child_part_names}

for child_part in child_part_names:
    for parent_part in parent_part_names:
        if knns[child_part][parent_part] is None:
            continue
        anchors = child_part_models[child_part].to_pcd(child_params[child_part])[
            knns[child_part][parent_part]
        ]
        targets_child[child_part][parent_part] = np.mean(
            anchors + deltas[child_part][parent_part], axis=1
        )
        targets_parent[child_part][parent_part] = parent_part_models[
            parent_part
        ].to_pcd(parent_params[parent_part])[
            target_indices[child_part][parent_part]
        ] 

child_part_targets = {}   
child_targets_viz = {}
parent_targets_viz = {}

for child_part in child_part_names:
    child_part_transform  = utils.pos_quat_to_transform(child_params[child_part].position, 
                                                  child_params[child_part].quat)
    for parent_part in parent_part_names:
        if knns[child_part][parent_part] is None:
            continue
        parent_part_transform  = utils.pos_quat_to_transform(parent_params[parent_part].position, 
                                                  parent_params[parent_part].quat)
        child_targets_viz =  child_targets_viz | {
                             f'child_targets_{child_part}_{parent_part}': \
                             utils.transform_pcd(targets_child[child_part][parent_part],
                                                 child_part_transform),
                             }
        parent_targets_viz = parent_targets_viz | {
                             f'parent_targets_{child_part}_{parent_part}': \
                             utils.transform_pcd(targets_parent[child_part][parent_part],
                                                 parent_part_transform),
                             }

viz_pcls = scene_pcls | child_targets_viz | parent_targets_viz
print(viz_pcls)
viz_utils.show_pcds_plotly(viz_pcls)

In [ ]:
# create ndf_interface

from src.ndf_interface import NDFPartInterface
interface = NDFPartInterface(
        canon_source_parts_paths = child_part_model_files,
        canon_target_parts_paths = parent_part_model_files,
        source_part_names = child_part_names,
        target_part_names= parent_part_names,)


In [ ]:
# extract relevant part pair relationships

possible_relevant_part_pairs = []
for child_part in child_part_names:
    part_pairs = []
    for parent_part in parent_part_names:
        if knns[child_part][parent_part] is None:
            continue
        part_pairs.append((child_part, parent_part))
    possible_relevant_part_pairs.append(part_pairs)

possible_constraint_programs = list(itertools.product(*possible_relevant_part_pairs))
print(possible_constraint_programs)

In [ ]:
# Make a prediction based on the training sample and calculate the distance between it and the ground-truth.
costs = []

child_parts = {name: scene_pcls[name] for name in child_part_names}
parent_parts = {name: scene_pcls[name] for name in parent_part_names}

for part_pair in possible_constraint_programs:
    print(part_pair)
    trans_predicted = interface.infer_relpose(
        child_parts, parent_parts, part_pair, se3=True, knn_pkl=knn_pickle_file
    )
    cost = utils.pose_distance(trans_predicted, ee_transform)
    costs.append(cost)
    print(cost)
    print()

min_cost = np.min(np.array(costs))
min_pair = possible_constraint_programs[np.argmin(np.array(costs))]

print(min_pair)
print(costs)
print(possible_constraint_programs)

In [ ]:
#save the interaction points